# Figure 4: Location and time distribution of collocaitons

To plot the flight paths, you need to download the **full dataset** available at the 4TU.ResearchData archive (see `../README.md`), and assign the path to the `collocations` folder to the `DATASET_LOCATION` variable. If `DATASET_LOCATION=None`, the script will plot everything except for the flight paths (in `fig04.png`).

In [ ]:
import random
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from iagos_toolkit.flight.aircraft_performance import create_flight_from_iagos

In [ ]:
# Set 'DATASET_LOCATION = None' to plot the figure without using the full dataset. 
# Setting it up to 'None' will not render the flight tracks in the left panel of the output figure.
# For Windows users, use the Windows native path format e.g. DATASET_LOCATION=r'C:\...\collocations'

DATASET_LOCATION = None # add path to 'collocations' folder
df = pd.read_csv('../data/landsat_sentinel_collocations_20260216.csv', comment='#')

In [ ]:
df = df[df["air_temperature_iagos_validity"] <= 0]
df = df[df["rhl_iagos_validity"] <= 0]
df = df[df["contrail_formation"].notna()]
df = df[df["efficiency_PS_IAGOS"].notna()]

df

In [ ]:
def split_on_antimeridian(lons, lats):
    """
    Splits a flight track whenever longitude jumps across the antimeridian.
    Returns list of (lon_segment, lat_segment)
    """
    lons = np.asarray(lons)
    lats = np.asarray(lats)

    # Find where |Δlon| > 180° (antimeridian crossing)
    jumps = np.where(np.abs(np.diff(lons)) > 180)[0]

    # If no jumps, return whole track
    if len(jumps) == 0:
        return [(lons, lats)]

    segments = []
    start = 0

    for j in jumps:
        # End the current segment at j
        segments.append((lons[start:j+1], lats[start:j+1]))
        start = j + 1

    # Last segment
    segments.append((lons[start:], lats[start:]))

    return segments

In [ ]:
# Creating the bar plot information
satellites = df["satellite"].tolist()
unique_sats = list(set(satellites))

years = pd.to_datetime(df["sensing_time"].to_list(), format="mixed").year
df_temp = pd.DataFrame({"year": years, "satellite": satellites})
df_temp["family"] = df_temp["satellite"].apply(lambda x: "Sentinel" if "Sentinel" in x else "Landsat")
counts = df_temp.groupby(["year", "family"]).size().reset_index(name="count")
pivot = counts.pivot(index="year", columns="family", values="count").fillna(0)

In [ ]:
# Styling configuration
satellite_color_map = {
    "Sentinel": "#E66100",
    "Landsat": "#5D3A9B"
}

sat_marker_map = {
    "Sentinel": "o",
    "Landsat": "x"
}

contrail_formation_map = {
    True: "#1A85FF",
    False: "#D41159"
}

# Create figure layout
fig = plt.figure(figsize=(12, 6), dpi=300)

# 1 × 3 grid: map spans first two columns, bar plot third column
gs = fig.add_gridspec(
    nrows=1,
    ncols=3,
    width_ratios=[1, 1, 1],
)

# Left: map
ax_map = fig.add_subplot(gs[0, :2], projection=ccrs.PlateCarree())

# Right: bar chart
ax_bar = fig.add_subplot(gs[0, 2])

# ----------------------------------------------------------
# (a) Map: collocations + flight tracks
ax_map.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax_map.set_global()

# Plot collocation points (randomized order)
data = list(zip(
    df["longitude"],
    df["latitude"],
    satellites,
    df["contrail_formation"]
))

random.shuffle(data)

for lon, lat, sat, contrail in data:
    ax_map.scatter(
        lon,
        lat,
        color=contrail_formation_map.get(contrail, "black"),
        marker=sat_marker_map.get(sat, "x"),
        s=30,
        alpha=0.8,
        transform=ccrs.PlateCarree(),
        zorder=5,
    )

# Plot flight tracks
if DATASET_LOCATION is not None:
    df_shuffled = df.sample(frac=1).reset_index(drop=True)

    for idx, row in df_shuffled.iterrows():
        try:
            with open(f"{DATASET_LOCATION}/{row['path']}/metadata.json", "r") as f:
                metadata = json.load(f)
        except Exception as e:
            print(f"Error loading metadata for row {idx}: {e}")
            continue

        flight = create_flight_from_iagos(
            f"{DATASET_LOCATION}/{row['path']}/{metadata['iagos_data']['filename']}",
            "A320",
            resample=True
        )

        lon = flight["longitude"]
        lat = flight["latitude"]

        segments = split_on_antimeridian(lon, lat)

        for seg_lon, seg_lat in segments:
            ax_map.plot(
                seg_lon,
                seg_lat,
                color="grey",
                alpha=0.2,
                zorder=4,
                transform=ccrs.PlateCarree()
            )

# Map legends
sat_handles = [
    ax_map.scatter([], [], color="black",
                   marker=sat_marker_map["Sentinel"],
                   label="Sentinel-2"),
    ax_map.scatter([], [], color="black",
                   marker=sat_marker_map["Landsat"],
                   label="Landsat"),
]

sat_legend = ax_map.legend(
    handles=sat_handles,
    title="Satellite",
    loc="lower left",
    fontsize=14,
    title_fontsize=14,
    framealpha=1.0
)

ax_map.add_artist(sat_legend)

# Contrail legend (color)
contrail_handles = [
    Patch(
        facecolor=contrail_formation_map[True],
        edgecolor="white",
        label="Yes"
    ),
    Patch(
        facecolor=contrail_formation_map[False],
        edgecolor="white",
        label="No"
    ),
]

ax_map.legend(
    handles=contrail_handles,
    title="Contrail formation",
    loc="lower right",
    fontsize=14,
    title_fontsize=14,
    framealpha=1.0
)

ax_map.text(
    0.05,
    1.08,
    "(a)",
    transform=ax_map.transAxes,
    ha="right",
    va="top",
    fontsize=14,
    fontweight="bold"
)

# -----------------------------------------
# (b) Dates: collocations per year

ax_bar.bar(
    pivot.index,
    pivot["Sentinel"],
    label="Sentinel-2",
    color=satellite_color_map["Sentinel"],
    edgecolor="black",
    hatch='//'
)

ax_bar.bar(
    pivot.index,
    pivot["Landsat"],
    bottom=pivot["Sentinel"],
    label="Landsat",
    color=satellite_color_map["Landsat"],
    edgecolor="black"
)

ax_bar.legend(fontsize=14, loc="upper left")

ax_bar.set_xlabel("Year", fontsize=14)
ax_bar.set_ylabel("Count", fontsize=14)
ax_bar.set_xticks([2012, 2014, 2016, 2018, 2020, 2022])
ax_bar.tick_params(axis="x", rotation=45, labelsize=14)
ax_bar.set_box_aspect(1.12)


# Panel label (b)
ax_bar.text(
    0.11,
    1.08,
    "(b)",
    transform=ax_bar.transAxes,
    ha="right",
    va="top",
    fontsize=14,
    fontweight="bold"
)


# Final layout + export
plt.tight_layout()
plt.savefig(
    "../figures/fig04.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.1
)
plt.show()